# Personalized Investment Recommendations

This notebook implements the Next Best Action (NBA) layer of the KYC pipeline.

**Prerequisites**: run all `utils/*.py` scripts first to populate `data/pickled_files/`.

The pipeline has two stages:
1. **Need prediction** — XGBoost, selected as best model for both targets, identifies which clients need Income or Accumulation products. Predictions are loaded from pre-computed pickles — no retraining needed.
2. **Product matching** — three approaches map the predicted need to a specific product:
   - **Baseline**: professor's risk-ceiling rule (highest product risk below `RiskPropensity`)
   - **Personalized**: a lifecycle-motivated `RiskTarget` score incorporating Age, Wealth, and FinancialEducation
   - **Confidence-based**: uses XGBoost predicted probabilities to decide *whether* to recommend at all

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


sys.path.insert(0, str(Path().resolve()))
from utils.preprocessing import (
    FEATURE_NAMES, TARGETS,
    build_features, load_data, load_result
)
from utils.next_best_action import (
    calculate_risk_target,
    evaluate_baseline_approach,
    evaluate_personalized_approach,
    evaluate_confidence_approach
)

# Load raw dataset — needed for RiskPropensity, Age, Wealth, FinancialEducation
df = load_data()
products_df = pd.read_excel(Path('Data') / 'Dataset2_Needs.xls', sheet_name='Products')
X = build_features(df)

print(f'Dataset : {df.shape[0]} clients, {X.shape[1]} features')
print(f'Products: {len(products_df)}')
products_df

## Load best models

XGBoost is used for both targets — it produces the best overall balance of
Precision and Recall, and unlike ensemble voting methods it outputs calibrated
probabilities, which are needed for the confidence-based recommendation layer.

In [ ]:
BEST_MODELS = {
    'IncomeInvestment':       'xgboost_shap',
    'AccumulationInvestment': 'xgboost_shap',
}

results = {}
for target, folder in BEST_MODELS.items():
    r = load_result(folder, target)
    results[target] = r
    tm = r['test_metrics']
    print(f"[{folder}] {target} — "
          f"F1: {tm['f1']:.3f}  "
          f"Precision: {tm['precision']:.3f}  "
          f"Recall: {tm['recall']:.3f}")

## Test-set metrics (reference table)

In [ ]:
rows = []
for target, r in results.items():
    row = {'Target': target, 'Model': r.get('model_name', BEST_MODELS[target])}
    row.update({k: round(v, 3) for k, v in r['test_metrics'].items()})
    rows.append(row)

pd.DataFrame(rows).set_index('Target')

## Build client prediction table

For each target, retrieve from the pickle: test-set predictions (0/1) and
predicted probabilities. Join with raw client features (Age, RiskPropensity,
FinancialEducation, Wealth) needed for product matching.

In [ ]:
client_frames = []

for target, r in results.items():
    y_true  = pd.Series(r['y_test_true'], name='y_true')
    y_pred  = pd.Series(r['y_test_pred'], name='y_pred')
    y_proba = pd.Series(r['y_test_proba'], name='proba')  # XGBoost always has probabilities

    idx = y_true.index
    frame = df.loc[idx, ['Age', 'RiskPropensity', 'FinancialEducation', 'Wealth']].copy()
    frame['y_true']  = y_true.values
    frame['y_pred']  = y_pred.values
    frame['proba']   = y_proba.values
    frame['target']  = target
    client_frames.append(frame)

clients = pd.concat(client_frames)
print('Predicted positives per target:')
print(clients[clients['y_pred'] == 1].groupby('target').size())

---
## Approach 1 — Baseline Risk Matching

For each client predicted to have a given need, recommend the product of the right type
with the **highest risk that stays below `RiskPropensity`**.

Simple, deterministic, MiFID-compliant by construction.
We implement it for both targets (the professor only did Accumulation).

In [ ]:
# Product arrays
prod_ids   = products_df['IDProduct'].values
prod_risks = products_df['Risk'].values
income_mask = products_df['Type'].values == 0   # Income products (Type=0)
accum_mask  = products_df['Type'].values == 1   # Accumulation products (Type=1)

type_mask = {
    'IncomeInvestment':       income_mask,
    'AccumulationInvestment': accum_mask,
}

# --- Baseline ---
nba_baseline = evaluate_baseline_approach(
    results, type_mask, df, prod_risks, prod_ids
)

print('Baseline coverage:')
for target in TARGETS:
    sub = nba_baseline[nba_baseline['Target'] == target]
    pct = sub['Matched'].mean() * 100
    print(f'  {target}: {sub["Matched"].sum()}/{len(sub)} ({pct:.1f}%)')

---
## Approach 2 — Personalized Risk Target

The baseline uses `RiskPropensity` as a hard ceiling — a single dimension.
We compute a **personalised risk target** combining four client dimensions:

```
RiskTarget = 0.50 * RiskPropensity        # primary MiFID signal
           + 0.20 * FinancialEducation     # literate clients can bear complexity
           - 0.20 * (Age / 100)            # older → shorter horizon → less risk
           + 0.10 * Wealth_norm            # wealthier → more resilient to losses
```

Weights are motivated by lifecycle theory (Merton 1969, Samuelson 1969).
The result is clipped to [0, 1] and matched to the **closest product** by risk —
not necessarily below the ceiling, but the most suitable given the full profile.

This is a discretised approximation of Markowitz mean-variance optimisation
over a finite product catalog.

In [ ]:
# --- Precalculate RiskTarget & execute Personalized ---
df = calculate_risk_target(df)

nba_personalized = evaluate_personalized_approach(
    results, type_mask, df, prod_risks, prod_ids
)

print('Personalized coverage:')
for target in TARGETS:
    sub = nba_personalized[nba_personalized['Target'] == target]
    print(f'  {target}: coverage {sub["Matched"].mean()*100:.1f}%'
          f'  |  avg gap {sub["Gap"].mean():.3f}')


---
## Comparison — Baseline vs Personalized

We compare the two approaches on three dimensions:
- **Coverage**: fraction of predicted-positive clients who receive a valid recommendation
- **Product diversification**: number of distinct products recommended
- **Suitability**: distribution of client risk vs product risk

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for row_idx, target in enumerate(TARGETS):
    base = nba_baseline[nba_baseline['Target'] == target]
    pers = nba_personalized[nba_personalized['Target'] == target]

    # Left: product distribution
    ax = axes[row_idx, 0]
    base_counts = base[base['Matched']]['RecommendedProd'].value_counts().sort_index()
    pers_counts = pers[pers['Matched']]['RecommendedProd'].value_counts().sort_index()
    all_prods = sorted(set(base_counts.index) | set(pers_counts.index))
    x = np.arange(len(all_prods))
    w = 0.35
    ax.bar(x - w/2, [base_counts.get(p, 0) for p in all_prods],
           width=w, label='Baseline', color='steelblue', alpha=0.8)
    ax.bar(x + w/2, [pers_counts.get(p, 0) for p in all_prods],
           width=w, label='Personalized', color='coral', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels([f'P{p}' for p in all_prods])
    ax.set_title(f'{target}\nProduct distribution')
    ax.set_ylabel('Clients')
    ax.legend()

    # Right: suitability scatter
    ax = axes[row_idx, 1]
    ax.scatter(base['ClientRisk'], base['ProductRisk'],
               alpha=0.3, s=15, label='Baseline', color='steelblue')
    ax.scatter(pers['ClientRisk'], pers['ProductRisk'],
               alpha=0.3, s=15, label='Personalized', color='coral')
    lim = 1.05
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.4, label='Equal risk')
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)
    ax.set_xlabel('Client RiskPropensity')
    ax.set_ylabel('Product Risk')
    ax.set_title(f'{target}\nSuitability')
    ax.legend(markerscale=2)

plt.tight_layout()
plt.show()

## Business insight summary

Three KPIs per target and approach:
- **Coverage**: % of predicted-positive clients who receive a valid recommendation
- **Products used**: number of distinct products recommended — a proxy for catalog diversification
- **Avg risk gap**: mean distance between `RiskTarget` and assigned product risk — how close we get to the ideal
- **Newly served**: clients the Baseline misses but Personalized reaches
- **Product gap**: clients unserved by both approaches — their `RiskPropensity` is below the minimum available product risk, signalling a gap in the product catalog

In [ ]:
print('=' * 65)
print('COVERAGE & DIVERSIFICATION SUMMARY')
print('=' * 65)

for target in TARGETS:
    base = nba_baseline[nba_baseline['Target'] == target]
    pers = nba_personalized[nba_personalized['Target'] == target]

    base_cov   = base['Matched'].mean() * 100
    pers_cov   = pers['Matched'].mean() * 100
    base_divs  = base[base['Matched']]['RecommendedProd'].nunique()
    pers_divs  = pers[pers['Matched']]['RecommendedProd'].nunique()
    pers_gap   = pers['Gap'].mean()

    # Clients newly served by Personalized (not reached by Baseline)
    base_unmatched = set(
        base[~base['Matched']]['ClientIdx']
    )
    pers_matched = set(
        pers[pers['Matched']]['ClientIdx']
    )
    newly_served = len(base_unmatched & pers_matched)

    print(f'\n{target}')
    print(f'  Coverage      — Baseline: {base_cov:.1f}%  |  Personalized: {pers_cov:.1f}%')
    print(f'  Products used — Baseline: {base_divs}       |  Personalized: {pers_divs}')
    print(f'  Avg risk gap  — Personalized: {pers_gap:.3f}')
    print(f'  Newly served by Personalized: {newly_served} clients')

print('\n' + '=' * 65)
print('PRODUCT GAP ANALYSIS')
print('Clients with no recommendation in either approach:')
for target in TARGETS:
    base = nba_baseline[nba_baseline['Target'] == target]
    pers = nba_personalized[nba_personalized['Target'] == target]
    unserved_base = set(base[~base['Matched']]['ClientIdx'])
    unserved_pers = set(pers[~pers['Matched']]['ClientIdx'])
    unserved_both = unserved_base & unserved_pers
    if unserved_both:
        avg_risk = df.loc[list(unserved_both), 'RiskPropensity'].mean()
        print(f'  {target}: {len(unserved_both)} clients, avg RiskPropensity = {avg_risk:.3f}')
        print(f'  → Product catalog gap: no product available below risk {avg_risk:.3f}')
    else:
        print(f'  {target}: all clients served')

---
## Approach 3 — Confidence-Based Recommendation

The previous two approaches answer *what* to recommend.
This approach answers *whether* to recommend at all.

The classifier outputs a probability score for each client — not just 0/1.
We use that score to segment clients into three tiers:

| Confidence | Probability | Action |
|---|---|---|
| High | > 0.75 | Direct recommendation — model is certain, advisor can proceed |
| Medium | 0.40 – 0.75 | Recommendation with caveat — suggest but flag for review |
| Low | < 0.40 | No recommendation — uncertainty too high, refer to human advisor |

This has direct MiFID II relevance: regulations require that recommendations
be appropriate and well-founded. Excluding low-confidence cases reduces
compliance risk and focuses advisor attention where the model is reliable.

The product recommendation for High and Medium tiers uses the Personalized
Risk Target logic — the best-performing matching approach.

In [ ]:
# --- Execution of Confidence-Based Approach ---
nba_confidence = evaluate_confidence_approach(
    results, type_mask, df, prod_risks, prod_ids
)

print('Confidence tier distribution:')
summary = (
    nba_confidence
    .groupby(['Target', 'ConfidenceTier'])
    .agg(
        Clients   = ('ClientIdx', 'count'),
        Matched   = ('Matched', 'sum'),
        Avg_proba = ('Proba', 'mean'),
    )
    .round(3)
)
print(summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, target in zip(axes, TARGETS):
    sub = nba_confidence[nba_confidence['Target'] == target]
    tier_order  = ['High', 'Medium', 'Low']
    tier_colors = {'High': '#2ecc71', 'Medium': '#f39c12', 'Low': '#e74c3c'}
    counts = sub['ConfidenceTier'].value_counts().reindex(tier_order, fill_value=0)
    bars = ax.bar(
        tier_order,
        counts.values,
        color=[tier_colors[t] for t in tier_order],
        alpha=0.85, edgecolor='white'
    )
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 3,
                str(val), ha='center', va='bottom', fontsize=11)
    ax.set_title(f'{target}\nClients by confidence tier')
    ax.set_ylabel('Clients')
    ax.set_xlabel('Confidence tier')

plt.tight_layout()
plt.show()
